# 7주차 ① 합성곱 연산의 원리 — 실습 1~4  〔교수용 정답본〕

> ⚠️ **이 파일은 교수용입니다.** 학생에게는 `11_conv_basics_blank.ipynb` 를 배포하세요.
> 빈칸 6곳 · 12줄이 모두 채워져 있습니다.

---

## 실습 1 — 커널을 손으로 적용 (9분)

**커널 하나가 "찾고 싶은 무늬" 하나**입니다.

```
   입력 5×5                커널 3×3            출력 3×3

   1 2 3 0 1              1 0 -1
   4 5 6 1 0      ⊛       1 0 -1       →      □ □ □
   7 8 9 0 1              1 0 -1              □ □ □
   1 1 1 1 1                                  □ □ □
   0 0 0 0 0
                   커널을 한 칸씩 옮기며
                   겹친 부분끼리 곱해서 더한다
```

위 커널은 **왼쪽이 밝고 오른쪽이 어두운 곳**, 즉 **세로 경계선**에서 큰 값을 냅니다.

> **그 커널 값을 사람이 정하지 않고 학습으로 찾는 것**이 CNN 입니다.
> 지금은 **우리가 숫자를 정해서** 넣어 보고, 어떤 일이 생기는지 눈으로 확인합니다.

In [ ]:
# 셀 1 — 이미지 한 장 준비  ★ 다운로드가 시작되면 그대로 두세요 (약 170MB)
import torch, torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

ROOT = "data"
train = datasets.CIFAR10(ROOT, train=True, download=True, transform=transforms.ToTensor())
CLASSES = ["비행기","자동차","새","고양이","사슴","개","개구리","말","배","트럭"]

img, label = train[7]
gray = img.mean(dim=0, keepdim=True).unsqueeze(0)      # (1,1,32,32) 흑백으로
print("입력 shape :", tuple(gray.shape), "| 클래스 :", CLASSES[label])

plt.figure(figsize=(2.6, 2.6))
plt.imshow(gray[0, 0], cmap="gray"); plt.axis("off"); plt.title(CLASSES[label])
plt.show()

In [ ]:
# 셀 2 — 커널 3종을 직접 적용
kernels = {
    "세로 경계": [[ 1, 0,-1], [ 1, 0,-1], [ 1, 0,-1]],
    "가로 경계": [[ 1, 1, 1], [ 0, 0, 0], [-1,-1,-1]],
    "흐리게":    [[1/9]*3, [1/9]*3, [1/9]*3],
}

fig, ax = plt.subplots(1, 4, figsize=(13, 3.2))
ax[0].imshow(gray[0, 0], cmap="gray"); ax[0].set_title("원본"); ax[0].axis("off")

for i, (name, k) in enumerate(kernels.items(), 1):
    w = torch.tensor(k, dtype=torch.float32).reshape(1, 1, 3, 3)

    out = F.conv2d(gray, w, padding=1)                  # ★ 합성곱 한 줄

    print(f"{name:8s} 출력 shape : {tuple(out.shape)}")
    ax[i].imshow(out[0, 0], cmap="gray"); ax[i].set_title(name); ax[i].axis("off")

plt.tight_layout(); plt.show()

> **관찰 포인트 ★**: 커널 숫자만 바꿨는데 **세로선만 남거나, 가로선만 남거나, 흐려집니다.**
> **커널 = 찾고 싶은 무늬**라는 말이 여기서 눈으로 확인됩니다.
>
> **지금은 우리가 9개 숫자를 정해서 넣었습니다.**
> CNN 은 이 숫자들을 **학습으로 스스로 찾습니다** —
> *"어떤 무늬를 찾아야 고양이를 잘 맞히는가"* 를 데이터로부터 배우는 것입니다.
> (1주차에 말한 *"특징을 사람이 설계하지 않는다"* 가 바로 이 이야기입니다.)

> **포인트**: `padding=1` 을 줘서 출력이 **32×32 로 유지**되는 것을 확인하세요.
> 다음 실습의 공식 그대로입니다.

---

## 실습 2 — 출력 크기 계산 ★ (7분)

```
                    입력 − 커널 + 2 × 패딩
        출력 크기 = ────────────────────────  + 1        (나눗셈은 내림)
                          스트라이드
```

> ### ★ 먼저 손으로 계산하세요. 셀을 실행하기 전에.
>
> | # | 입력 | 커널 | 스트라이드 | 패딩 | 출력은? |
> |---|---|---|---|---|---|
> | 1 | 32 | 3 | 1 | 1 | ? |
> | 2 | 32 | 3 | 1 | 0 | ? |
> | 3 | 32 | 3 | 2 | 1 | ? |
> | 4 | 32 | 5 | 1 | 2 | ? |
>
> **네 개를 다 하지 않아도 됩니다. 두 개만 해 보고 넘어가세요.**
> 순서를 바꾸면(먼저 실행하면) 아무도 계산하지 않습니다.

In [ ]:
# 셀 3 — 손으로 계산하고, 코드로 확인한다
x = torch.randn(1, 3, 32, 32)          # 배치1, 채널3, 32×32

설정 = [
    dict(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1),
    dict(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=0),
    dict(in_channels=3, out_channels=16, kernel_size=3, stride=2, padding=1),
    dict(in_channels=3, out_channels=16, kernel_size=5, stride=1, padding=2),
]

for cfg in 설정:
    계산 = (32 - cfg["kernel_size"] + 2*cfg["padding"]) // cfg["stride"] + 1
    실제 = nn.Conv2d(**cfg)(x).shape

    맞음 = "○" if 계산 == 실제[-1] else "✗"
    print(f"k={cfg['kernel_size']} s={cfg['stride']} p={cfg['padding']} "
          f"| 손계산 {계산:2d} | 실제 {tuple(실제)}  {맞음}")

> **관찰 포인트 ★**: 출력 shape 은 `(1, 16, H, W)` 입니다.
> **채널이 3 → 16 으로 바뀐 것**에 주목하세요.
> `out_channels=16` 은 **커널이 16개**라는 뜻이고, 출력 채널 하나하나가 **커널 하나가 만든 특징맵**입니다.

> **핵심 ★**: `k=3, s=1, p=1` 은 **32 → 32**, `k=5, s=1, p=2` 도 **32 → 32** 입니다.
> 규칙: **`padding = (kernel_size − 1) / 2` 이면 크기가 유지**됩니다. 실무에서 계속 씁니다.
>
> 그래서 실무 CNN 은 **크기 유지 조합을 기본으로 쓰고, 크기 줄이기는 풀링에 맡깁니다.**

> **함정 ★**: 나눗셈은 **내림**입니다. `stride=2` 에서 홀수가 나오면 잘립니다.
> `(32 − 3 + 2) / 2 + 1 = 16.5 → 16`.
> **2교시에서 FC 입력 크기를 계산할 때 여기서 틀리면 오류가 납니다.**

---

## 실습 3 — 풀링을 직접 확인한다 (5분)

```
   MaxPool2d(2)          2×2 창에서 가장 큰 값만 남긴다

     1  3 │ 2  0                4 │ 2
     4  2 │ 1  1      →         ─┼─      32×32  →  16×16
     ─────┼─────                4 │ 5
     0  4 │ 5  1
     1  1 │ 2  3
```

| | 하는 일 | 왜 |
|---|---|---|
| **MaxPool** | 창에서 최댓값 | *"이 근처에 그 무늬가 있었다"* 만 남긴다. **CNN 의 기본** |
| AvgPool | 창에서 평균 | 부드럽게. 마지막 층에서 가끔 |

**직접 확인할 것은 두 가지입니다** — 값이 어떻게 고르는지, 그리고 **파라미터가 몇 개인지.**

In [ ]:
# 셀 4 — 풀링 : 파라미터가 없는 축소 연산
y = torch.tensor([[[[1., 3., 2., 0.],
                    [4., 2., 1., 1.],
                    [0., 4., 5., 1.],
                    [1., 1., 2., 3.]]]])
print("입력 4×4 :\n", y[0, 0])

pool_max = nn.MaxPool2d(2)
pool_avg = nn.AvgPool2d(2)

print("\nMaxPool2d(2) 결과 2×2 :\n", pool_max(y)[0, 0])
print("\nAvgPool2d(2) 결과 2×2 :\n", pool_avg(y)[0, 0])
print("\n풀링의 파라미터 수 :", sum(p.numel() for p in pool_max.parameters()),
      "  ← 학습하지 않는다")

> **관찰 포인트 ★★**: **파라미터가 0개입니다.**
> 풀링은 학습하는 것이 아니라 **그냥 줄이는 연산**입니다.

| 풀링의 효과 | 설명 |
|---|---|
| 크기를 줄인다 | 계산량·메모리 감소 |
| **위치 변화에 둔감해진다** ★ | 고양이가 조금 움직여도 같은 답 |
| 시야가 넓어진다 | 뒤쪽 층의 3×3 커널이 원본에서는 더 넓은 영역을 본다 |

> 그래서 **`Conv → BN → ReLU → Pool`** 을 한 묶음으로 쌓습니다 — **2교시의 기본 블록**입니다.

---

## 실습 4 — 특징맵 시각화 ★ (10분)

**층이 깊어질수록 무엇을 보는지** 눈으로 확인합니다.

> **forward hook** 이란 — 층에 *"이 층을 지나갈 때 나를 불러 줘"* 라고 등록해 두는 장치입니다.
> 모델 코드를 고치지 않고 **중간 결과를 꺼낼 수 있습니다.**
> (9주차 Grad-CAM 도 같은 원리로 동작합니다.)

In [ ]:
# 셀 5 — 학습 안 한 CNN 의 특징맵
torch.manual_seed(0)
net = nn.Sequential(
    nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(),      # 1층  <- net[0]
    nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(),     # 2층  <- net[3]
)
print(net)

feats = {}
def hook(name):
    def fn(m, i, o): feats[name] = o.detach()      # ★ detach 를 빼면 그래프가 붙어 남는다
    return fn

h1 = net[0].register_forward_hook(hook("1층"))
h2 = net[3].register_forward_hook(hook("2층"))

x = train[7][0].unsqueeze(0)          # (1,3,32,32)
_ = net(x)

for name in ["1층", "2층"]:
    f = feats[name][0]
    print(f"{name} 특징맵 shape : {tuple(f.shape)}")

    fig, ax = plt.subplots(1, 8, figsize=(14, 2))
    for i in range(8):
        ax[i].imshow(f[i], cmap="viridis"); ax[i].axis("off")

    fig.suptitle(f"{name} 특징맵 (앞 8개 채널)"); plt.show()

h1.remove(); h2.remove()              # ★ hook 은 쓰고 나면 떼어 낸다

> **관찰 포인트 ★**: **채널마다 반응하는 곳이 다릅니다.**
> 어떤 채널은 윤곽에, 어떤 채널은 배경에 밝게 반응합니다.
> **아직 학습 전인데도** 그렇습니다 — 커널이 무작위여도 서로 다른 무늬를 잡기 때문입니다.

> **포인트**: 2층 특징맵이 **16×16** 인 것을 확인하세요 — 사이의 `MaxPool2d(2)` 때문입니다.
> 채널은 **8 → 16** 으로 늘었습니다.

> **핵심 메시지 ★★**: **층이 깊어질수록 보는 것이 달라집니다.**
> ```
>   1층  : 선, 색 경계 같은 단순한 것
>   2층  : 그 선들이 모인 모양 (모서리, 질감)
>   더 깊은 층 : 눈, 바퀴 같은 부분  →  결국 물체 전체
> ```
> **사람이 이 단계를 설계하지 않았습니다.** 데이터에서 스스로 나온 것이고,
> 이것이 딥러닝의 정의적 특징(1주차)입니다.

> **막히면**: hook 이 안 걸리면 층 인덱스(`net[0]`, `net[3]`)를 확인하세요.
> `print(net)` 으로 번호를 볼 수 있습니다.

In [ ]:
# 셀 6 (참고) — 배포된 도우미로 같은 일을 더 짧게
from feature_map_hook import FeatureCatcher, show_feature_maps, show_kernels, conv_out_size

with FeatureCatcher(net, {"1층": net[0], "2층": net[3]}) as c:
    net(x)                            # ★ with 를 벗어나면 hook 이 자동으로 떨어진다

show_feature_maps(c.feats["1층"], "1층")
show_kernels(net[0])                  # 커널 자체도 그려 본다 (학습 전이라 잡음처럼 보인다)

print("\n출력 크기 공식 검산 :", conv_out_size(32, kernel=3, stride=2, padding=1))

> **포인트**: `with` 블록을 벗어나면 hook 이 **자동으로 떨어집니다.**
> 직접 `register_forward_hook` 을 쓰면 **반드시 `remove()`** 해야 합니다 —
> 안 떼면 계속 쌓여서 메모리를 먹습니다.

> **`show_kernels`** 로 **커널 자체**도 볼 수 있습니다. 지금은 학습 전이라 **잡음처럼** 보입니다.
> 2교시에서 학습시킨 뒤 다시 보면 달라집니다 — 여유가 있으면 비교해 보세요.

---

## 1교시 핵심 정리

| 개념 | 한 줄 정의 |
|------|-----------|
| **MLP 의 한계** ★★ | ① 공간 정보 소실 ② 파라미터 폭증 |
| **합성곱** | 작은 커널을 미끄러뜨리며 곱해 더한다. **가중치 공유** |
| **커널** | 찾고 싶은 무늬 하나. **값은 학습으로 찾는다** |
| **`out_channels`** | **커널의 개수** = 출력 특징맵의 채널 수 |
| **출력 크기** ★★ | `(입력 − 커널 + 2×패딩) / 스트라이드 + 1` (내림) |
| **크기 유지 조합** | `kernel=3, padding=1` (일반화: `padding=(k−1)/2`) |
| **풀링** | **파라미터 0개**. 크기 축소 + **위치 변화에 둔감** |
| **특징맵** | 층이 깊어질수록 단순한 것 → 복잡한 것 |

### 반드시 외우기 ★

```python
nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1)   # 크기 유지
nn.MaxPool2d(2)                                                 # 절반으로

# 출력 크기 = (입력 − 커널 + 2×패딩) // 스트라이드 + 1
```

### 1교시 체크리스트

- [ ] MLP 가 이미지에 불리한 이유 **두 가지**를 말할 수 있다 ★★
- [ ] 커널을 직접 바꿔 가며 필터 결과가 달라지는 것을 봤다
- [ ] `out_channels` 가 **커널 개수**라는 것을 안다
- [ ] **출력 크기를 손으로 계산**하고 실제와 대조했다 ★
- [ ] `padding=(k−1)/2` 이면 크기가 유지되는 것을 안다
- [ ] 풀링에 **파라미터가 0개**라는 것과 그 목적을 안다
- [ ] 1층·2층 특징맵을 보고 **shape 차이**를 설명할 수 있다

### 저장

`Kernel → Restart Kernel and Run All Cells` 로 다시 실행하고,
**출력이 살아 있는 상태로** `11_conv_basics.ipynb` 로 저장합니다.

> **다음 시간 예고**: 재료가 준비됐습니다 — `Conv2d`, `MaxPool2d`, 그리고 출력 크기 공식.
> 2교시에는 이걸 **어떤 순서로 몇 번 쌓을지**를 배우고(LeNet → VGG → ResNet),
> **CIFAR-10 분류 CNN 을 직접 작성**합니다.
> 그리고 5·6주차의 MLP 와 **정면으로 비교**합니다.